# ISM 6564 — Week 2 practice: vectors, weighting, and distance

**Ungraded. Nothing to submit.**

Week 2's graded assignment asks you to build a similarity-search engine over a
128-document help-centre corpus and defend a choice of representation and
metric. This notebook rehearses the *mechanics* that assignment leans on, at a
scale small enough that you can check every number by hand:

1. Build a term-document matrix by hand and make `CountVectorizer` reproduce it.
2. Build TF-IDF by hand and make `TfidfVectorizer` reproduce it — including the
   two `+1`s and the L2 normalisation that sklearn applies and the textbook
   formula does not.
3. Compute six similarity/distance measures on the same pair of vectors.
4. Implement Levenshtein edit distance.
5. Prove to yourself what L2 normalisation does to cosine, dot product, and
   Euclidean distance.
6. Find a case where raw dot product and cosine **rank documents differently**.
7. Use Jaccard on shingles for near-duplicate detection.
8. Watch Euclidean distance lose its discriminating power in high dimensions.

Every step ends with a check cell that tells you whether you got it right, so
you can work the whole notebook without opening the solution.

In [2]:
import random

import numpy as np
import pandas as pd
import torch
from scipy.spatial import distance
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 120)
print(f"torch device: {device}")

torch device: cpu


---

## Step 1 — A term-document matrix, by hand and by sklearn

Five toy documents. They are already lowercased and space-separated, so
"tokenize" here means nothing more than `str.split()`. That is deliberate: it
takes the tokenizer out of the argument so we can concentrate on the matrix.

In [3]:
DOCS = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "cats and dogs are pets",
    "the cat and the dog are friends",
    "a log is not a mat",
]
for i, d in enumerate(DOCS, start=1):
    print(f"d{i}: {d}")

d1: the cat sat on the mat
d2: the dog sat on the log
d3: cats and dogs are pets
d4: the cat and the dog are friends
d5: a log is not a mat


**Your turn.** Build two things with plain Python:

- `vocab` — the sorted list of distinct tokens across all five documents.
- `M` — a NumPy integer array of shape `(5, len(vocab))` where `M[i, j]` is the
  number of times `vocab[j]` occurs in `DOCS[i]`.

Sorted order matters: `CountVectorizer` sorts its vocabulary alphabetically, and
we are about to compare against it element by element.

In [ ]:
from collections import Counter

# TODO: build the sorted vocabulary as a list of strings.
vocab = sorted({tok for doc in DOCS for tok in doc.split()})
term_index={ t:j for j, t in enumerate(vocab)}

# TODO: build the (5, len(vocab)) count matrix as a NumPy int array.
M = np.zeros((len(DOCS), len(vocab)), dtype=np.int64)
for i, doc in enumerate(DOCS):
    for tok, n in Counter(doc.split()).items():
        M[i,term_index[tok]]=n

print(f"{len(DOCS)} documents x {len(vocab)} terms")
pd.DataFrame(M, index=[f"d{i}" for i in range(len(DOCS))], columns=vocab)


5 documents x 16 terms


,a,and,are,cat,cats,dog,dogs,friends,is,log,mat,not,on,pets,sat,the
d0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,1,2
d1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,1,2
d2,0,1,1,0,1,0,1,0,0,0,0,0,0,1,0,0
d3,0,1,1,1,0,1,0,1,0,0,0,0,0,0,0,2
d4,2,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0


In [5]:
if vocab is None or M is None:
    print("Step 1 not done yet — fill in the TODOs above, then re-run.")
else:
    print(f"vocab ({len(vocab)} terms): {vocab}")
    print()
    print(pd.DataFrame(M, index=[f"d{i}" for i in range(1, 6)], columns=vocab))

vocab (16 terms): ['a', 'and', 'are', 'cat', 'cats', 'dog', 'dogs', 'friends', 'is', 'log', 'mat', 'not', 'on', 'pets', 'sat', 'the']

    a  and  are  cat  cats  dog  dogs  friends  is  log  mat  not  on  pets  sat  the
d1  0    0    0    1     0    0     0        0   0    0    1    0   1     0    1    2
d2  0    0    0    0     0    1     0        0   0    1    0    0   1     0    1    2
d3  0    1    1    0     1    0     1        0   0    0    0    0   0     1    0    0
d4  0    1    1    1     0    1     0        1   0    0    0    0   0     0    0    2
d5  2    0    0    0     0    0     0        0   1    1    1    1   0     0    0    0


Now make sklearn produce the identical matrix.

`CountVectorizer` does its own tokenizing with a regex, `token_pattern`, whose
default is `r"(?u)\b\w\w+\b"` — **two or more word characters**. Our `d5`
contains the single-character token `a`, which that pattern silently drops. To
get sklearn to agree with a `str.split()` tokenization you have to replace the
tokenizer *and* switch the pattern off:

```python
CountVectorizer(tokenizer=str.split, token_pattern=None)
```

Passing `tokenizer=` on its own warns
`"The parameter 'token_pattern' will not be used since 'tokenizer' is not None"`
and leaves a dead parameter behind; passing neither gives you a different
vocabulary than the one you just built by hand.

In [6]:
# TODO: fit a CountVectorizer that reproduces `M` exactly, and store the
# dense array in X_sk.
from sklearn.feature_extraction.text import CountVectorizer

cv_default= CountVectorizer()
cv_default.fit(DOCS)

cv = CountVectorizer(tokenizer=str.split,token_pattern=None)
X_sk = cv.fit_transform(DOCS).toarray()

print(f"default token_pattern     -> {len(cv_default.get_feature_names_out()):2d} terms")
print(f"tokenizer=str.split       -> {len(cv.get_feature_names_out()):2d} terms")
print("dropped by the default    ->",
      sorted(set(cv.get_feature_names_out()) - set(cv_default.get_feature_names_out())))

# assert list(cv.get_feature_names_out()) == vocab
# assert np.array_equal(X_sk.toarray(), M)
# print("\nhand-built matrix == CountVectorizer output   (asserted)")

# # get X_sk to a dense array
# print(type(X_sk))
# print(f"shape {X_sk.shape}, stored non-zeros{X_sk.nnz} of {np.prod(X_sk.shape)} cells")
# print(f"\nd1 as a dense row: {X_sk[1].toarray()[0]}")
# print("d1 as (term,count) pairs:",
#       {cv.get_feature_names_out()[j]: int(v) for j, v in zip(X_sk[1].indices, X_sk[1].data)})


default token_pattern     -> 15 terms
tokenizer=str.split       -> 16 terms
dropped by the default    -> ['a']


In [7]:
if X_sk is None or M is None:
    print("Step 1 not finished — fill in the TODOs above, then re-run.")
else:
    same_vocab = list(cv.get_feature_names_out()) == list(vocab)
    same_counts = np.array_equal(np.asarray(M), X_sk)
    print(f"vocabularies identical  : {same_vocab}")
    print(f"count matrices identical: {same_counts}")
    assert same_vocab and same_counts, "not yet matching — check the tokenizer arguments"

    # And the default settings, for contrast.
    cv_default = CountVectorizer().fit(DOCS)
    dropped = sorted(set(vocab) - set(cv_default.get_feature_names_out()))
    print()
    print(f"default CountVectorizer vocabulary size: {len(cv_default.vocabulary_)} "
          f"(ours: {len(vocab)})")
    print(f"tokens the default pattern threw away  : {dropped}")

vocabularies identical  : True
count matrices identical: True

default CountVectorizer vocabulary size: 15 (ours: 16)
tokens the default pattern threw away  : ['a']


---

## Step 2 — TF-IDF by hand, matching sklearn exactly

The textbook formula is

$$\mathrm{tf\text{-}idf}(t, d) = \mathrm{tf}(t, d)\cdot\log\frac{N}{\mathrm{df}(t)}$$

`TfidfVectorizer` does **not** compute that. With its default settings
(`smooth_idf=True`, `sublinear_tf=False`, `norm="l2"`) it computes

$$\mathrm{idf}(t) = \ln\!\left(\frac{1 + N}{1 + \mathrm{df}(t)}\right) + 1,
\qquad
w(t,d) = \mathrm{tf}(t,d)\cdot \mathrm{idf}(t),
\qquad
\mathbf{w}_d \leftarrow \frac{\mathbf{w}_d}{\lVert \mathbf{w}_d \rVert_2}$$

Three differences: natural log, `+1` inside *and* outside, and every row
rescaled to unit length. The trailing `+1` means **no term ever gets a weight of
zero**, even one that appears in every single document.

Reproduce sklearn's numbers with NumPy.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

N = len(DOCS)

# TODO: document frequency for each vocabulary term -- how many of the 5
# documents contain it at least once. Shape (len(vocab),).



df = (X_sk >0).sum(axis=0)


# TODO: sklearn's smoothed idf.
idf = np.log((1+N)/(1 + df)) +1



# TODO: tf * idf, then L2-normalise each row. `normalize(A, norm="l2")` from
# sklearn.preprocessing does the row-wise part for you.
tf=X_sk
tfidf_manual = normalize(tf*idf, norm="l2", axis=1)




In [9]:
tv = TfidfVectorizer(tokenizer=str.split, token_pattern=None)
tfidf_sk = tv.fit_transform(DOCS).toarray()

if tfidf_manual is None:
    print("Step 2 not done yet — fill in the TODOs above, then re-run.")
else:
    max_diff = np.abs(np.asarray(tfidf_manual) - tfidf_sk).max()
    print(f"max |manual - sklearn| = {max_diff:.3e}")
    print(f"idf vectors match      = {np.allclose(idf, tv.idf_)}")
    assert np.allclose(tfidf_manual, tfidf_sk), "not matching yet"
    print()
    comparison = pd.DataFrame({
        "df": df,
        "sklearn idf": idf,
        "textbook log(N/df)": np.log(N / df),
    }, index=vocab)
    print(comparison.round(6))

max |manual - sklearn| = 1.110e-16
idf vectors match      = True

         df  sklearn idf  textbook log(N/df)
a         1     2.098612            1.609438
and       2     1.693147            0.916291
are       2     1.693147            0.916291
cat       2     1.693147            0.916291
cats      1     2.098612            1.609438
dog       2     1.693147            0.916291
dogs      1     2.098612            1.609438
friends   1     2.098612            1.609438
is        1     2.098612            1.609438
log       2     1.693147            0.916291
mat       2     1.693147            0.916291
not       1     2.098612            1.609438
on        2     1.693147            0.916291
pets      1     2.098612            1.609438
sat       2     1.693147            0.916291
the       3     1.405465            0.510826


---

## Step 3 — Six ways to compare two vectors

Take `d1` ("the cat sat on the mat") and `d2` ("the dog sat on the log"). They
share `the`, `sat`, `on` and differ in two content words.

Compute, using `scipy.spatial.distance` where a function exists:

| Quantity | On which vectors |
|---|---|
| cosine **distance** and cosine **similarity** | TF-IDF rows |
| dot product | TF-IDF rows |
| Euclidean distance | TF-IDF rows |
| Manhattan (cityblock) distance | TF-IDF rows |
| Minkowski distance with `p=3` | TF-IDF rows |
| Jaccard **similarity** | binary presence/absence |
| Hamming distance | binary presence/absence |

`scipy` returns *distances*. `distance.jaccard` gives the Jaccard **distance**,
so the similarity is `1 - distance.jaccard(...)`. `distance.hamming` returns the
**fraction** of positions that differ, not the count.

In [10]:
# Given: the two vector pairs you will compare.
a_tfidf, b_tfidf = tfidf_sk[0], tfidf_sk[1]
a_bin = b_bin = None
if X_sk is not None:
    a_bin, b_bin = X_sk[0] > 0, X_sk[1] > 0
print(f"tf-idf rows: shape {a_tfidf.shape}, both L2-normalised "
      f"(norm a = {np.linalg.norm(a_tfidf):.6f}, norm b = {np.linalg.norm(b_tfidf):.6f})")

tf-idf rows: shape (16,), both L2-normalised (norm a = 1.000000, norm b = 1.000000)


In [11]:
# TODO: fill in each measure. Use scipy where it has the function.
measures = {
    "cosine distance":    distance.cosine(a_tfidf, b_tfidf),
    "cosine similarity":  1-distance.cosine(a_tfidf,b_tfidf),
    "dot product":        a_tfidf @ b_tfidf,
    "euclidean":          distance.euclidean(a_tfidf,b_tfidf),
    "cityblock":          distance.cityblock(a_tfidf, b_tfidf),
    "minkowski p=3":      distance.minkowski(a_tfidf,b_tfidf,p=3),
    "jaccard similarity": distance.jaccard(a_bin,b_bin),
    "hamming distance":   distance.hamming(a_bin,b_bin),
}

In [12]:
if any(v is None for v in measures.values()):
    print("Step 3 not done yet — fill in the TODOs above, then re-run.")
else:
    for name, value in measures.items():
        print(f"{name:<20} {value:.6f}")
    print()
    # Two identities worth internalising.
    print("cosine similarity == dot product for L2-normalised rows:",
          np.isclose(measures["cosine similarity"], measures["dot product"]))
    print("sklearn cosine_similarity agrees with 1 - scipy cosine distance:",
          np.isclose(cosine_similarity([a_tfidf], [b_tfidf])[0, 0],
                     measures["cosine similarity"]))
    n_terms = len(vocab)
    print(f"hamming {measures['hamming distance']:.4f} "
          f"= {round(measures['hamming distance'] * n_terms)}/{n_terms} positions differ")

cosine distance      0.296024
cosine similarity    0.703976
dot product          0.703976
euclidean            0.769447
cityblock            1.538894
minkowski p=3        0.610711
jaccard similarity   0.571429
hamming distance     0.250000

cosine similarity == dot product for L2-normalised rows: True
sklearn cosine_similarity agrees with 1 - scipy cosine distance: True
hamming 0.2500 = 4/16 positions differ


---

## Step 4 — Levenshtein edit distance

The measures so far all compare *vectors*. Edit distance compares *strings*
directly: the minimum number of single-character insertions, deletions, or
substitutions that turns one string into the other. It is what powers "did you
mean…?", fuzzy joins on customer names, and OCR cleanup.

Fill in the dynamic-programming recurrence. `D[i][j]` is the edit distance
between the first `i` characters of `a` and the first `j` characters of `b`:

$$D[i][j] = \min\begin{cases}
D[i-1][j] + 1 & \text{(delete)}\\
D[i][j-1] + 1 & \text{(insert)}\\
D[i-1][j-1] + \mathbb{1}[a_i \neq b_j] & \text{(substitute or match)}
\end{cases}$$

In [13]:
def levenshtein(a: str, b: str) -> int:
    """Minimum single-character edits turning `a` into `b`."""
    m, n = len(a), len(b)
    D = [[0] * (n + 1) for i in range(m + 1)]
  # TODO: initialise the first column (D[i][0]) and the first row (D[0][j]).
    for i in range(1, m+1):
        D[i][0] = i
    for j in range(1, n+1):
        D[0][j] = j

    # TODO: fill the rest of the table with the recurrence above
    for i in range(1, m+1):
        for j in range (1, n+1):
            D[i][j]=min(
                D[i-1][j]+1,
                D[i][j-1]+1,
                D[i-1][j-1] + (a[i-1]!= b[j-1]),
            )
  


    return D[m][n]


levenshtein_done = True   # TODO: set to True once you have implemented it



In [14]:
EXPECTED = [
    ("waranty", "warranty", 1),   # one insertion
    ("kitten", "sitting", 3),     # the textbook example
    ("recieve", "receive", 2),    # a transposition costs 2, not 1
    ("cat", "cats", 1),
    ("", "abc", 3),
    ("same", "same", 0),
]

if not levenshtein_done:
    print("Step 4 not done yet — implement levenshtein(), then re-run.")
else:
    for a, b, want in EXPECTED:
        got = levenshtein(a, b)
        flag = "ok " if got == want else "BAD"
        print(f"{flag} levenshtein({a!r:>10}, {b!r:>10}) = {got}  (expected {want})")
    assert all(levenshtein(a, b) == w for a, b, w in EXPECTED)

ok  levenshtein( 'waranty', 'warranty') = 1  (expected 1)
ok  levenshtein(  'kitten',  'sitting') = 3  (expected 3)
ok  levenshtein( 'recieve',  'receive') = 2  (expected 2)
ok  levenshtein(     'cat',     'cats') = 1  (expected 1)
ok  levenshtein(        '',      'abc') = 3  (expected 3)
ok  levenshtein(    'same',     'same') = 0  (expected 0)


---

## Step 5 — What L2 normalisation actually does

For any two **unit-length** vectors $\mathbf{a}$ and $\mathbf{b}$:

$$\lVert \mathbf{a} - \mathbf{b}\rVert_2^2
= \mathbf{a}\cdot\mathbf{a} - 2\,\mathbf{a}\cdot\mathbf{b} + \mathbf{b}\cdot\mathbf{b}
= 2\bigl(1 - \cos(\mathbf{a},\mathbf{b})\bigr)$$

Squared Euclidean distance is a *strictly decreasing* function of cosine
similarity. So on L2-normalised vectors, ranking by cosine descending, by dot
product descending, and by Euclidean distance ascending all produce the **same
order**.

Verify the identity on every pair of the five TF-IDF rows, then verify the
ranking consequence.

In [16]:
# TODO: for every pair (i, j) with i < j among the 5 TF-IDF rows, compute
#   lhs = squared Euclidean distance
#   rhs = 2 * (1 - cosine similarity)
# and record the largest absolute difference between them.

max_gap = 0.0

for i in range(5):
    for j in range(i+1):
        lhs= distance.euclidean(tfidf_sk[i],tfidf_sk[j])**2
        rhs=2*(1-distance.cosine(tfidf_sk[i],tfidf_sk[j]))

        gap= abs(lhs-rhs)
        if gap > max_gap:
            max_gap==gap

In [17]:
if max_gap is None:
    print("Step 5 not done yet — fill in the TODO above, then re-run.")
else:
    print(f"max |d^2 - 2(1-cos)| over all 10 pairs = {max_gap:.3e}")
    # Float arithmetic will not give you exact zero. Compare with a tolerance.
    assert max_gap < 1e-12, "identity does not hold — check your formula"

    # The ranking consequence, on a query against the five documents.
    q = tv.transform(["cat mat"]).toarray()[0]
    cos_rank = np.argsort(-cosine_similarity(tfidf_sk, q.reshape(1, -1)).ravel(),
                          kind="stable")
    dot_rank = np.argsort(-(tfidf_sk @ q), kind="stable")
    euc_rank = np.argsort([distance.euclidean(row, q) for row in tfidf_sk],
                          kind="stable")
    print()
    print("query 'cat mat'")
    print(f"  cosine    order: {[f'd{i+1}' for i in cos_rank]}")
    print(f"  dot       order: {[f'd{i+1}' for i in dot_rank]}")
    print(f"  euclidean order: {[f'd{i+1}' for i in euc_rank]}")
    print(f"  all three identical: "
          f"{list(cos_rank) == list(dot_rank) == list(euc_rank)}")

max |d^2 - 2(1-cos)| over all 10 pairs = 0.000e+00

query 'cat mat'
  cosine    order: ['d1', 'd4', 'd5', 'd2', 'd3']
  dot       order: ['d1', 'd4', 'd5', 'd2', 'd3']
  euclidean order: ['d1', 'd4', 'd5', 'd2', 'd3']
  all three identical: True


---

## Step 6 — Where dot product and cosine disagree

Add a sixth document: `d2` written out three times. It says nothing new — it is
the same content at triple the length. Then query it with `"log mat"`.

Compute, on **raw counts** (not TF-IDF), the dot product and the cosine
similarity between the query and each of the six documents.

Before you run it, predict: which document *should* rank first for `"log mat"`?

In [18]:
DOCS6 = DOCS + [" ".join([DOCS[1]] * 3)]
for i, d in enumerate(DOCS6, start=1):
    print(f"d{i} ({len(d.split()):>2} tokens): {d}")

d1 ( 6 tokens): the cat sat on the mat
d2 ( 6 tokens): the dog sat on the log
d3 ( 5 tokens): cats and dogs are pets
d4 ( 7 tokens): the cat and the dog are friends
d5 ( 6 tokens): a log is not a mat
d6 (18 tokens): the dog sat on the log the dog sat on the log the dog sat on the log


In [27]:
QUERY = "log mat"

# TODO: fit a CountVectorizer on DOCS6 (same arguments as Step 1), transform
# the query, and compute both scores against all six documents.
cv6 = CountVectorizer(tokenizer=str.split, token_pattern=None)
X6 = cv6.fit_transform(DOCS6).toarray()           # (6, V) raw counts
q6 = cv6.transform([QUERY]).toarray()[0]           # (V,)   raw counts for the query
dot_scores = q6.dot(X6.T)    # (6,)
cos_scores = 1- distance.cdist(q6.reshape(1, -1), X6, "cosine")[0]   # (6,)

In [28]:
if dot_scores is None or cos_scores is None:
    print("Step 6 not done yet — fill in the TODOs above, then re-run.")
else:
    table = pd.DataFrame({
        "tokens": X6.sum(axis=1),
        "dot": dot_scores,
        "cosine": cos_scores,
    }, index=[f"d{i}" for i in range(1, 7)])
    print(table.round(6))
    print()
    print("dot    ranking: "
          f"{[f'd{i+1}' for i in np.argsort(-np.asarray(dot_scores), kind='stable')]}")
    print("cosine ranking: "
          f"{[f'd{i+1}' for i in np.argsort(-np.asarray(cos_scores), kind='stable')]}")
    print()
    print(f"dot    puts d6 above d5: {dot_scores[5] > dot_scores[4]}")
    print(f"cosine puts d5 above d6: {cos_scores[4] > cos_scores[5]}")

    tokens  dot  cosine
d1       6    1    0.25
d2       6    1    0.25
d3       5    0    0.00
d4       7    0    0.00
d5       6    2    0.50
d6      18    3    0.25

dot    ranking: ['d6', 'd5', 'd1', 'd2', 'd3', 'd4']
cosine ranking: ['d5', 'd1', 'd2', 'd6', 'd3', 'd4']

dot    puts d6 above d5: True
cosine puts d5 above d6: True


---

## Step 7 — Jaccard on shingles: near-duplicate detection

Cosine on TF-IDF tells you two documents are about the same topic. It does
**not** tell you one is a lightly edited copy of the other — and a policy page
and its revision share a topic *and* share their exact phrasing.

The standard tool is Jaccard similarity over **shingles**: overlapping
$n$-grams of the document. Word order is preserved inside each shingle, so two
documents built from the same sentences score high while two documents merely
about the same subject do not.

Implement `shingles(text, n)` returning the set of space-joined word $n$-grams,
then compute Jaccard on the two policy revisions below.

In [33]:
A = "the item must be unworn unwashed and carry its original hang tags"
B = "the item must be unworn unwashed and still carry the original hang tags"
print("A:", A)
print("B:", B)

A: the item must be unworn unwashed and carry its original hang tags
B: the item must be unworn unwashed and still carry the original hang tags


In [34]:

def shingles(text: str, n: int = 3) -> set:
    """Set of overlapping word n-grams, each joined by a single space."""
    # TODO: implement.
    words= text.split()
    if len(words) >= n:
        return set(" ".join(words[i:i+n]) for i in range(len(words)-n+1))
    return set()


def jaccard(s1: set, s2: set) -> float:
    """|intersection| / |union|; 0.0 when both sets are empty."""
    # TODO: implement.
    if not s1 and not s2:
        return 0.0
    intersection = len(s1 & s2)
    union_size=len(s1 | s2)
    return intersection/union_size 


shingles_done = True  # TODO: set to True once both are implemented

In [35]:
if not shingles_done:
    print("Step 7 not done yet — implement the two functions, then re-run.")
else:
    sa, sb = shingles(A, 3), shingles(B, 3)
    print(f"|shingles(A)| = {len(sa)}   |shingles(B)| = {len(sb)}")
    print(f"|A and B| = {len(sa & sb)}   |A or B| = {len(sa | sb)}")
    print(f"3-shingle Jaccard      = {jaccard(sa, sb):.6f}")
    print(f"bag-of-words   Jaccard = {jaccard(set(A.split()), set(B.split())):.6f}")
    print()
    for n in (1, 2, 3, 5, 8):
        print(f"  n={n}: jaccard = {jaccard(shingles(A, n), shingles(B, n)):.4f}")
    assert abs(jaccard(sa, sb) - 0.4) < 1e-9, "expected 0.4 for n=3 on these two strings"
    assert jaccard(shingles(A, 3), shingles(A, 3)) == 1.0

|shingles(A)| = 10   |shingles(B)| = 11
|A and B| = 6   |A or B| = 15
3-shingle Jaccard      = 0.400000
bag-of-words   Jaccard = 0.846154

  n=1: jaccard = 0.8462
  n=2: jaccard = 0.5333
  n=3: jaccard = 0.4000
  n=5: jaccard = 0.2143
  n=8: jaccard = 0.0000


---

## Step 8 — Distance concentration in high dimensions

The "curse of dimensionality" claim you will hear is that in high dimensions all
points become roughly equidistant, so distance stops discriminating. Measure it
rather than believing it.

For each dimensionality `d`, draw 500 standard-normal points, compute all
pairwise distances, and report the **relative contrast**

$$\text{contrast} = \frac{d_{\max} - d_{\min}}{\bar{d}}$$

A contrast near 0 means the farthest pair is barely farther apart than the
nearest pair — at which point "nearest neighbour" is close to meaningless.

In [ ]:
from scipy.spatial.distance import pdist
DIMS = [2, 10, 100, 1000, 10000] # list of dims
records=[]# set dataframe


for d in DIMS:
    X= rng.standard_normal((500,d))# 500 standard-normal points

    euc_dists=pdist(X, metric="euclidean")
    cos_dists = pdist(X, metric="cosine")

    euc_contrast= (euc_dists.max() - euc_dists.min()) / euc_dists.min()
    cos_contrast = (cos_dists.max() - cos_dists.min()) / cos_dists.min()

    records.append({
        "d": d,
        "euclidean": euc_contrast,
        "cosine": cos_contrast
    })

contrast = pd.DataFrame(records).set_index("d")
print(contrast)
# TODO: for each d in DIMS, sample `rng.standard_normal((500, d))`, compute all
# pairwise euclidean and cosine distances with `distance.pdist`, and record the
# relative contrast for each. Build a DataFrame indexed by d with columns
# "euclidean" and "cosine".


         euclidean        cosine
d                               
2      1425.555781  5.316617e+09
10        9.092566  3.276608e+01
100       0.966621  1.439309e+00
1000      0.209666  3.325849e-01
10000     0.060941  9.293121e-02


In [39]:
if contrast is None:
    print("Step 8 not done yet — fill in the TODO above, then re-run.")
else:
    print("relative contrast (dmax - dmin) / dmean, 500 random points")
    print(contrast.round(4))

relative contrast (dmax - dmin) / dmean, 500 random points
       euclidean        cosine
d                             
2      1425.5558  5.316617e+09
10        9.0926  3.276610e+01
100       0.9666  1.439300e+00
1000      0.2097  3.326000e-01
10000     0.0609  9.290000e-02


---

## Where this lands

| You practised | The graded assignment uses it for |
|---|---|
| Steps 1–2: term-document matrix, TF-IDF that matches sklearn | Task 1 — building and characterising four representations |
| Steps 3–4: six measures, edit distance | Task 2 — a `score_matrix()` parameterised by metric |
| Step 5: the L2 identity | Task 3 — explaining why three metrics tie on TF-IDF |
| Step 6: dot vs cosine reversal | Task 4a — measuring document-length bias on real data |
| Step 7: shingle Jaccard | Task 4b — the near-duplicate audit |
| Step 8: distance concentration | Task 6 — the honest version of the argument |

Everything here was done on five toy documents you can check by hand. The graded
assignment runs the same machinery over 128 documents and 28 judged queries,
where you can no longer eyeball the answer — which is exactly why the evaluation
harness in Task 3 carries the most marks.